# 03 · The core experiment

Extreme closing moves, split by whether closing volume was abnormally heavy, and
what happens overnight.

**The benchmark matters.** Equities drift up overnight, so a positive
conditional mean after an up close is not evidence of persistence.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from closingbell import config as C, calendar_utils as cal

def table(name):
    return pd.read_csv(C.TABLES / f"{name}.csv")

sessions = pd.read_parquet(C.PROCESSED / "sessions.parquet")
print(f"{len(sessions):,} ticker-sessions, {sessions.session.min()} to {sessions.session.max()}")

15,756 ticker-sessions, 2021-01-04 to 2026-03-31


In [2]:
from closingbell import events as ev
universe = ev.event_panel(sessions)
extreme = universe[universe.is_extreme]
print(f"eligible sessions : {len(universe):,}")
print(f"5% tail events    : {len(extreme):,}  ({len(extreme)/len(universe):.1%})")
print(f"mean |close30| in tails: {extreme.r_close30.abs().mean()*1e4:.1f} bp")
print()
print(f"UNCONDITIONAL overnight mean: {universe.r_overnight.mean()*1e4:+.2f} bp")
print(f"P(overnight > 0)            : {(universe.r_overnight > 0).mean():.3f}")

eligible sessions : 14,958
5% tail events    : 1,974  (13.2%)
mean |close30| in tails: 76.9 bp

UNCONDITIONAL overnight mean: +4.81 bp
P(overnight > 0)            : 0.538


## The central 2x2

`excess_bps` is the cell mean minus the unconditional drift — the column that actually answers the question.

In [3]:
t = table('central_2x2_overnight')
t[["direction", "volume_regime", "n", "mean_close30_bps", "mean_bps",
   "ci_low_bps", "ci_high_bps", "excess_bps", "excess_ci_low_bps",
   "excess_ci_high_bps", "excess_p_value", "p_same_sign"]].round(2)

,direction,volume_regime,n,mean_close30_bps,mean_bps,ci_low_bps,ci_high_bps,excess_bps,excess_ci_low_bps,excess_ci_high_bps,excess_p_value,p_same_sign
0,all (baseline),all,14958,-0.16,4.81,-0.19,9.85,NaN,NaN,NaN,NaN,NaN
1,strong up,high,625,83.77,7.21,-13.71,28.14,2.41,-17.04,22.08,0.81,0.53
2,strong up,ordinary,391,58.78,2.87,-13.06,19.44,-1.94,-17.57,13.63,0.79,0.53
3,strong down,high,611,-85.50,29.34,5.66,51.59,24.53,2.78,45.47,0.03,0.39
4,strong down,ordinary,347,-69.60,18.01,-3.31,39.00,13.20,-7.26,33.04,0.21,0.39


Strong **up** closes land within a few basis points of the drift: no persistence,
no reversal. Strong **down** closes recover +18 to +29 bp overnight — a partial
reversal of roughly a quarter to a third of the closing decline.

## Does closing volume separate the two?

In [4]:
vc = table('volume_contrast_all')
vc[["direction", "outcome", "n_high", "n_ordinary", "mean_high_bps",
    "mean_ordinary_bps", "diff_bps", "diff_ci_low_bps", "diff_ci_high_bps",
    "boot_p_value"]].round(2)

,direction,outcome,n_high,n_ordinary,mean_high_bps,mean_ordinary_bps,diff_bps,diff_ci_low_bps,diff_ci_high_bps,boot_p_value
0,strong up,r_overnight,625,391,7.21,2.87,4.35,-17.72,27.43,0.69
1,strong down,r_overnight,611,347,29.34,18.01,11.33,-16.39,38.58,0.40
2,strong up,r_open30_next,624,390,-7.82,4.59,-12.40,-26.09,1.08,0.07
3,strong down,r_open30_next,610,346,5.34,3.65,1.68,-14.02,17.97,0.85
4,strong up,r_session_next,625,391,0.17,-9.59,9.76,-22.20,40.92,0.53
5,strong down,r_session_next,611,347,10.75,-11.96,22.71,-5.98,52.88,0.12
6,strong up,r_next_close_to_close,625,391,6.91,-6.60,13.51,-22.70,48.91,0.44
7,strong down,r_next_close_to_close,611,347,39.39,6.36,33.03,-3.05,69.29,0.08


No contrast clears conventional significance at any horizon. The point estimates lean toward heavier volume meaning *more* reversal after down closes — the opposite of a conviction story — but the data does not support that reading either.

## Every horizon

In [5]:
ct = table('central_2x2')
ct[["outcome", "direction", "volume_regime", "n", "mean_bps", "ci_low_bps",
    "ci_high_bps", "t_stat_clustered"]].round(2)

,outcome,direction,volume_regime,n,mean_bps,ci_low_bps,ci_high_bps,t_stat_clustered
0,r_overnight,all (baseline),all,14958,4.81,-0.19,9.85,1.90
1,r_overnight,strong up,high,625,7.21,-13.71,28.14,-0.25
2,r_overnight,strong up,ordinary,391,2.87,-13.06,19.44,-1.07
3,r_overnight,strong down,high,611,29.34,5.66,51.59,3.47
4,r_overnight,strong down,ordinary,347,18.01,-3.31,39.00,2.18
5,r_open30_next,all (baseline),all,14927,2.60,-0.09,5.31,1.82
6,r_open30_next,strong up,high,624,-7.82,-19.80,4.35,-0.34
7,r_open30_next,strong up,ordinary,390,4.59,-6.39,15.48,1.18
8,r_open30_next,strong down,high,610,5.34,-6.79,18.11,0.75
9,r_open30_next,strong down,ordinary,346,3.65,-7.98,14.30,0.51


## Deciles

In [6]:
d = table('overnight_by_decile')
d[["decile", "n", "mean_close30_bps", "mean_bps", "ci_low_bps", "ci_high_bps",
   "p_positive"]].round(2)

,decile,n,mean_close30_bps,mean_bps,ci_low_bps,ci_high_bps,p_positive
0,1,1465,-69.39,20.22,4.61,34.94,0.59
1,2,1467,-35.76,8.16,-2.32,18.35,0.56
2,3,1399,-22.34,10.50,1.56,19.79,0.55
3,4,1490,-12.79,2.81,-6.25,11.67,0.54
4,5,1502,-4.45,7.41,-1.78,16.78,0.56
5,6,1516,3.65,1.65,-5.53,8.89,0.53
6,7,1469,11.94,-1.05,-9.04,6.57,0.52
7,8,1443,20.64,-4.50,-13.04,3.69,0.50
8,9,1452,32.15,-3.65,-14.15,7.08,0.50
9,10,1755,62.24,6.34,-5.53,18.85,0.51


In [7]:
dv = table('overnight_by_decile_volume')
dv.pivot(index="decile", columns="volume_regime", values="mean_bps").round(1)

volume_regime,high,ordinary
decile,,
1,24.2,14.9
2,17.5,3.1
3,11.8,10.0
4,6.3,1.7
5,18.3,3.8
6,14.2,-2.1
7,-3.3,-0.3
8,-5.0,-4.3
9,-2.5,-4.1


## Persistence, against the right benchmark

In [8]:
pb = table('persistence_by_decile_pooled')
pb["excess_pp"] = 100 * pb.excess_over_benchmark
pb[["decile", "n", "p_persistent", "benchmark", "excess_pp", "ci_low", "ci_high"]].round(3)

,decile,n,p_persistent,benchmark,excess_pp,ci_low,ci_high
0,1,1465,0.406,0.462,-5.560,0.381,0.431
1,2,1467,0.433,0.462,-2.889,0.408,0.458
2,3,1399,0.447,0.462,-1.434,0.421,0.474
3,4,1485,0.444,0.464,-1.916,0.419,0.470
4,5,1461,0.444,0.479,-3.460,0.419,0.470
5,6,1465,0.509,0.518,-0.904,0.483,0.534
6,7,1456,0.513,0.536,-2.305,0.487,0.539
7,8,1443,0.500,0.538,-3.855,0.474,0.525
8,9,1452,0.500,0.538,-3.825,0.474,0.526
9,10,1755,0.514,0.538,-2.429,0.491,0.537


The realised persistence rate sits **below** the drift-implied benchmark in every
decile — a broad, mild reversal tendency across the whole distribution rather
than something that switches on in the tails.

See `results/figures/fig05`, `fig06` (hero) and `fig07`.